# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ErenSnowh/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Binary classification**, used as a scoring/ranking tool.

The decision this improves: *out of thousands of content pages, which ones should an editor review for refresh first?* That's a prioritization problem. The most direct way to build a priority queue is to classify each page as "declining" or "not declining" and use the model's predicted probability as a confidence score to rank the queue.

I considered pure ranking (learn-to-rank), but classification is the right starting point here because:
- The label is naturally binary — a page is either declining or it isn't
- The predicted probability gives us a ranking for free
- It's interpretable: a content editor can understand "78% chance this page is declining" better than an abstract rank score

This maps to the framing skill's table: "Will this one decline?" → Classification → yes/no label from an observed outcome → precision@K.

In [1]:
# No code needed for this section — pure framing.
# The backing numbers come in the sections below.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is `is_declining_label`, which equals 1 when `trend_direction == "down"`.

This label comes from an **observed outcome**: a page's impressions dropped by more than 20% between the previous 30-day window and the most recent 30-day window. It's measuring something that actually happened in the real world (search visibility dropped), not something a human decided to flag.

Critical leakage note: `trend_direction` and `trend_pct` can **never** be model features — they are the label source. Using them would mean the model is just looking up the answer. The data dictionary flags this explicitly, and the pipeline in `scripts/ml_utils.py` excludes them.

Also excluded from features: `content_id` and `client_id` (pseudonymous identifiers for grouping/splits only), and any FlyRank product flags like `health_score` (these encode downstream decisions — using them would be circular).

In [2]:
import pandas as pd
import os

# Find repo root (works in both Colab and local)
if os.path.exists('../../data/raw/content_refresh_anonymized.csv'):
    ROOT = '../..'
elif os.path.exists('data/raw/content_refresh_anonymized.csv'):
    ROOT = '.'
else:
    raise FileNotFoundError('Cannot find data — run from repo root or work/notebooks/')

df = pd.read_csv(f'{ROOT}/data/raw/content_refresh_anonymized.csv')

# Build the target label the same way the pipeline does
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print('Label distribution:')
print(df['is_declining_label'].value_counts())
print(f'\nBase rate (% declining): {df["is_declining_label"].mean():.1%}')
print(f'Total rows: {len(df):,}')

Label distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Base rate (% declining): 54.2%
Total rows: 30,000


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: `precision@50`** — precision in the top 50% of the ranked queue.

Why this metric and not accuracy or F1?

- The output feeds a **human review queue**. An editor picks up the top-ranked pages and works through them. What matters is: "of the pages I'm reviewing, how many actually need a refresh?" That's precision at the top of the queue.
- Accuracy is misleading here because the classes are roughly balanced (54% declining). A model that predicts "declining" for everything gets 54% accuracy — useless but looks decent.
- The existing hand-written rules achieve **0.24 precision@50** (from `outputs/model_report.md`). That means ~76% of the pages the rules flag as top-priority are false alarms. Beating 0.24 is the bar.

**Secondary metric: ROC-AUC** — measures overall discrimination ability across all thresholds. Useful for comparing models but not directly tied to the editorial workflow.

What "good" looks like: precision@50 above 0.50 would mean more than half the editor's queue is actually declining pages — a meaningful improvement over the 0.24 baseline that would save real review hours.

In [3]:
# The baseline to beat
base_rate = df['is_declining_label'].mean()
baseline_p50 = 0.24  # from outputs/model_report.md

print(f'Base rate (random queue precision): {base_rate:.1%}')
print(f'Rule baseline precision@50:         {baseline_p50}')
print(f'\nA random queue gives ~{base_rate:.0%} precision.')
print(f'The hand-written rules give 0.24 precision@50.')
print(f'The bar: beat 0.24 with a learned model.')

Base rate (random queue precision): 54.2%
Rule baseline precision@50:         0.24

A random queue gives ~54% precision.
The hand-written rules give 0.24 precision@50.
The bar: beat 0.24 with a learned model.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page** (pseudonymized). The dataset has 30,000 content pages across 32 pseudonymized clients, each with trailing 90-day search and engagement metrics.

Key columns by role:
- **Identifiers** (not features): `content_id`, `client_id`
- **Search metrics**: `impressions_90d`, `clicks_90d`, `avg_position`, `ctr`, `days_with_impressions`
- **Engagement metrics**: `sessions_90d`, `engaged_sessions_90d`, `scroll_rate`, `engagement_rate`
- **Content properties**: `word_count`, `content_age_days`, `days_since_last_update`, `content_type`
- **Target**: `is_declining_label` (derived from `trend_direction`)
- **Never features**: `trend_direction`, `trend_pct` (label source)

In [4]:
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Clients: {df["client_id"].nunique()}')
print(f'\nOne row = one content page\n')

# Show key columns for a few rows
key_cols = [
    'content_id', 'client_id',
    'impressions_90d', 'clicks_90d', 'avg_position', 'ctr',
    'sessions_90d', 'word_count', 'content_age_days',
    'content_type', 'trend_direction', 'is_declining_label'
]
df[key_cols].head(5)

Shape: 30,000 rows × 45 columns
Clients: 32

One row = one content page



,content_id,client_id,impressions_90d,clicks_90d,avg_position,ctr,sessions_90d,word_count,content_age_days,content_type,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,29,10.6,0.76,17,3221.0,187,keyword article,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,20.3,0.05,9,2481.0,445,keyword article,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,0.09,11,3515.0,141,keyword article,down,1
3,content_331d6c4de07b,client_19581e27de,11751,58,6.2,0.49,78,NaN,463,keyword article,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,44.0,0.13,145,2803.0,263,keyword article,down,1


In [5]:
# Quick data profile
print('Missing values in key columns:')
missing = df[key_cols].isnull().sum()
print(missing[missing > 0].to_string())
print(f'\nNo missing values' if missing.sum() == 0 else '')

print(f'\nContent types: {df["content_type"].value_counts().to_dict()}')
print(f'Trend distribution: {df["trend_direction"].value_counts().to_dict()}')

Missing values in key columns:
word_count    7699


Content types: {'keyword article': 27207, 'feedly article': 2096, 'comparison article': 697}
Trend distribution: {'down': 16262, 'stable': 5962, 'up': 4388, 'new': 2236, 'flat': 1152}


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

FlyRank already runs hand-written rules in production — health scores, quick-win tags, needs-attention flags. These are if-this-then-that thresholds chosen by hand. They work for the obvious cases, but the rule baseline only achieves **0.24 precision@50**. Three reasons a learned model can do better:

1. **Too many interacting signals.** There are 44 columns in this dataset. Whether a page is declining depends on impressions, position, CTR, engagement, content age, word count, content type, and more — all at once. A rule like "flag if impressions dropped > 20%" ignores position changes, seasonal patterns, content type differences, and engagement shifts. Writing one if-statement per interaction explodes combinatorially.

2. **The best threshold shifts.** What counts as "declining" for a high-traffic keyword article (10,000+ impressions) is different from a feedly article with 50 impressions. A fixed threshold misses this. A model learns per-segment decision boundaries from the data.

3. **The cost of false positives is real.** At 0.24 precision, 76% of the pages an editor reviews under the current rules don't actually need refreshing. That's wasted editorial hours. A model that lifts precision to 0.68 (the random forest result from the pipeline) means the editor's queue is nearly 3× more useful.

The code below shows what happens with a simple single-rule approach — it illustrates why one threshold can't capture the pattern.

In [6]:
# Demo: a simple rule vs the complexity of the real pattern
# Rule: "flag everything with impressions drop" (impressions_last_30d < impressions_prev_30d)

df['simple_rule_flag'] = (
    df['impressions_last_30d'] < df['impressions_prev_30d']
).astype(int)

flagged = df[df['simple_rule_flag'] == 1]
rule_precision = flagged['is_declining_label'].mean() if len(flagged) > 0 else 0

print(f'Simple rule: flag if impressions_last_30d < impressions_prev_30d')
print(f'Pages flagged: {len(flagged):,} out of {len(df):,} ({len(flagged)/len(df):.1%})')
print(f'Precision of this rule: {rule_precision:.2f}')
print(f'\nCompare to:')
print(f'  Rule baseline precision@50: 0.24')
print(f'  Random forest precision@50: 0.68 (from model_report.md)')
print(f'\nThe simple rule flags too many pages and still misses the nuance.')
print(f'A model that uses position, engagement, content age, and word count')
print(f'together can separate real declines from noise — that\'s why ML helps.')

# Clean up temp column
df.drop(columns=['simple_rule_flag'], inplace=True)

Simple rule: flag if impressions_last_30d < impressions_prev_30d
Pages flagged: 19,716 out of 30,000 (65.7%)
Precision of this rule: 0.82

Compare to:
  Rule baseline precision@50: 0.24
  Random forest precision@50: 0.68 (from model_report.md)

The simple rule flags too many pages and still misses the nuance.
A model that uses position, engagement, content age, and word count
together can separate real declines from noise — that's why ML helps.


## The one-paragraph frame

Putting it together using the framing skill template:

> For **a content editor at FlyRank**, deciding **which content pages to review and refresh first**, we will build a **binary classifier that outputs a ranked priority queue** from **30,000 pseudonymized content pages with 90-day search and engagement metrics**, predicting **`is_declining_label` (observed impression decline > 20% between 30-day windows)**, measured by **precision@50** (precision in the top half of the queue). A wrong call costs **wasted editor hours reviewing pages that don't need refreshing** (false positive) or **missing a page that's quietly losing traffic** (false negative). A plain rule isn't enough because **44 features interact in ways that shift across content types and traffic levels — the hand-written rules achieve only 0.24 precision@50, while a random forest reaches 0.68**. We will claim only **observed, directional, and decision-support** results.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.